# CIFAR-100 Results

This notebook visualizes CIFAR-100 runs produced by `run_cifar100_experiments.sh`.

By default it loads the most recent folder under `cifar100_runs/`. To inspect a specific run, set `EXPERIMENT_DIR` in the first code cell.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()

# Set this to a concrete path to inspect a specific experiment.
# Example: EXPERIMENT_DIR = ROOT / "cifar100_runs" / "cifar100_seed_0_20260601_120000"
EXPERIMENT_DIR = None

if EXPERIMENT_DIR is None:
    runs_root = ROOT / "cifar100_runs"
    candidates = sorted(
        [path for path in runs_root.glob("cifar100_seed_*") if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
    )
    if not candidates:
        raise FileNotFoundError("No CIFAR-100 experiment folders found under cifar100_runs/.")
    EXPERIMENT_DIR = candidates[-1]
else:
    EXPERIMENT_DIR = Path(EXPERIMENT_DIR)

print(f"Using experiment: {EXPERIMENT_DIR}")
print((EXPERIMENT_DIR / "manifest.txt").read_text() if (EXPERIMENT_DIR / "manifest.txt").exists() else "No manifest found.")

## Evaluation Metrics

In [ ]:
METHOD_ORDER = [
    "MRL",
    "MRL-E",
    "BOR-MRL matrix_exp",
    "Independent-block BOR-MRL",
    "BOR-MRL frozen",
    "BOR-MRL cayley",
    "BOR-MRL householder",
    "Full feature",
    "Fixed 512",
]
METHOD_RANK = {method: idx for idx, method in enumerate(METHOD_ORDER)}

RUN_LABELS = {
    "mrl": "MRL",
    "mrle": "MRL-E",
    "bor_mrl": "BOR-MRL matrix_exp",
    "bor_block_mrl": "Independent-block BOR-MRL",
    "bor_mrl_frozen": "BOR-MRL frozen",
    "bor_mrl_cayley": "BOR-MRL cayley",
    "bor_mrl_householder": "BOR-MRL householder",
    "full_feature": "Full feature",
}

def method_label(run_name, payload):
    if run_name in RUN_LABELS:
        return RUN_LABELS[run_name]
    if payload.get("bor_block_mrl"):
        return "Independent-block BOR-MRL"
    if payload.get("bor_mrl"):
        if payload.get("bor_mode") == "frozen":
            return "BOR-MRL frozen"
        return f"BOR-MRL {payload.get('bor_orthogonal_map', 'orthogonal')}"
    if payload.get("mrl") and payload.get("efficient"):
        return "MRL-E"
    if payload.get("mrl"):
        return "MRL"
    if run_name == "full_feature":
        return "Full feature"
    if run_name.startswith("fixed_"):
        return f"Fixed {run_name.split('_', 1)[1]}"
    return run_name

def method_sort_key(method):
    return METHOD_RANK.get(method, len(METHOD_RANK))

records = []
eval_dir = EXPERIMENT_DIR / "eval"
for metrics_path in sorted(eval_dir.glob("*.json")):
    payload = json.loads(metrics_path.read_text())
    run_name = metrics_path.stem
    for row in payload["metrics"]:
        records.append({
            "run": run_name,
            "method": method_label(run_name, payload),
            "method_rank": method_sort_key(method_label(run_name, payload)),
            "rep_size": int(row["rep_size"]),
            "top1": float(row["top1"]),
            "top5": float(row["top5"]),
            "bor_mode": payload.get("bor_mode"),
            "bor_orthogonal_map": payload.get("bor_orthogonal_map"),
            "num_images": int(payload["num_images"]),
            "total_time": float(payload["total_time"]),
            "seed": int(payload["seed"]),
            "deterministic": bool(payload["deterministic"]),
        })

eval_df = pd.DataFrame(records)
if eval_df.empty:
    raise FileNotFoundError(f"No metrics JSON files found in {eval_dir}")

eval_df = eval_df.sort_values(["method_rank", "run", "rep_size"]).reset_index(drop=True)
eval_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

for method in sorted(eval_df["method"].unique(), key=method_sort_key):
    group = eval_df[eval_df["method"] == method].sort_values("rep_size")
    axes[0].plot(group["rep_size"], group["top1"] * 100, marker="o", label=method)
    axes[1].plot(group["rep_size"], group["top5"] * 100, marker="o", label=method)

for ax, title, ylabel in zip(axes, ["Top-1 Accuracy", "Top-5 Accuracy"], ["Top-1 (%)", "Top-5 (%)"]):
    ax.set_xscale("log", base=2)
    ax.set_xlabel("Representation size")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()


## Best Evaluation Result Per Run

In [ ]:
best_df = eval_df.loc[eval_df.groupby("run")["top1"].idxmax()].copy()
best_df["top1_percent"] = best_df["top1"] * 100
best_df["top5_percent"] = best_df["top5"] * 100
best_df.sort_values(["method_rank", "top1_percent"], ascending=[True, False])[["run", "method", "rep_size", "top1_percent", "top5_percent", "num_images", "seed", "deterministic"]]


## Saved Checkpoints


In [ ]:
checkpoint_dir = EXPERIMENT_DIR / "checkpoints"
checkpoint_records = []

if checkpoint_dir.exists():
    for checkpoint_path in sorted(checkpoint_dir.glob("*_final_weights.pt")):
        run_name = checkpoint_path.name.removesuffix("_final_weights.pt")
        latest_path = checkpoint_dir / f"{run_name}_latest_weights.pt"
        checkpoint_records.append({
            "run": run_name,
            "method": method_label(run_name, {}),
            "checkpoint": checkpoint_path,
            "size_mb": checkpoint_path.stat().st_size / (1024 ** 2),
            "latest_checkpoint": latest_path if latest_path.exists() else None,
        })
else:
    for checkpoint_path in sorted((EXPERIMENT_DIR / "trainlogs").glob("*/final_weights.pt")):
        run_name = checkpoint_path.parent.name
        latest_path = checkpoint_path.parent / "latest_weights.pt"
        checkpoint_records.append({
            "run": run_name,
            "method": method_label(run_name, {}),
            "checkpoint": checkpoint_path,
            "size_mb": checkpoint_path.stat().st_size / (1024 ** 2),
            "latest_checkpoint": latest_path if latest_path.exists() else None,
        })

checkpoint_df = pd.DataFrame(checkpoint_records)
if checkpoint_df.empty:
    raise FileNotFoundError(f"No saved checkpoints found under {EXPERIMENT_DIR}")

checkpoint_df.sort_values("method", key=lambda values: values.map(method_sort_key))


## Training Logs

In [ ]:
def load_log(log_path):
    rows = []
    with open(log_path) as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def largest_top1_column(frame):
    nested_cols = [col for col in frame.columns if col.startswith("top_1_")]
    if nested_cols:
        return sorted(nested_cols, key=lambda col: int(col.rsplit("_", 1)[1]))[-1]
    return "top_1" if "top_1" in frame.columns else None

logs = {}
for run_dir in sorted((EXPERIMENT_DIR / "trainlogs").iterdir()):
    log_path = run_dir / "log"
    if log_path.exists():
        logs[run_dir.name] = load_log(log_path)

print(f"Loaded logs for {len(logs)} run(s): {', '.join(logs)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for run_name, frame in logs.items():
    if "epoch" not in frame:
        continue
    if "train_loss" in frame:
        axes[0].plot(frame["epoch"], frame["train_loss"], marker="o", label=run_name)
    top1_col = largest_top1_column(frame)
    if top1_col is not None:
        axes[1].plot(frame["epoch"], frame[top1_col] * 100, marker="o", label=f"{run_name} ({top1_col})")

axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_title("Validation Top-1 During Training")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Top-1 (%)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()

## Export Summary

In [ ]:
summary_path = EXPERIMENT_DIR / "cifar100_eval_summary.csv"
eval_df.to_csv(summary_path, index=False)
summary_path